# Vault Auto-Unseal with AWS KMS — On-Prem Vault accessing AWS KMS

This notebook demonstrates how to configure Vault's `seal "awskms"` when **Vault runs on-premises** (on a K8s cluster outside of AWS) and needs to access an **AWS KMS key** for auto-unseal.

### Authentication with AWS STS AssumeRole
The Vault AWS KMS seal reads long-lived bootstrap credentials from a shared credentials file and uses them only to call `sts:AssumeRole`. The resulting short-lived STS credentials authorize KMS operations and are refreshed by the AWS credential provider.

### Least-privilege separation
The bootstrap IAM user can only assume `vault-kms-unseal`. The role, not the user, receives `kms:Encrypt`, `kms:Decrypt`, and `kms:DescribeKey`. Kubernetes mounts the bootstrap credentials as an AWS shared credentials file; they are never exposed as environment variables.

### Architecture
```
┌─ On-Prem (K8s Cluster) ─────────┐      ┌─ AWS Account (KMS) ─────────────────┐
│                                  │      │                                      │
│  Vault Pod                       │      │  IAM user: vault-autounseal          │
│   ├─ shared credentials profile  │      │   └─ sts:AssumeRole only             │
│   └─ seal "awskms"              │ STS  │                                      │
│        role_arn ──────────────────┼─────▶│  Role: vault-kms-unseal             │
│        kms_key_id ────────────────┼─────▶│   └─ KMS Encrypt/Decrypt/Describe   │
│                                  │      │                                      │
│  Secret mounted as a file;       │      │  KMS Key: vault-auto-unseal          │
│  STS credentials are temporary   │      │   └─ Key policy grants the role      │
│  and refreshed automatically.    │      │                                      │
└──────────────────────────────────┘      └──────────────────────────────────────┘
```

The static key cannot access KMS directly. Vault must successfully assume the role before auto-unseal can use the KMS key.

### What Vault actually passes to the AWS KMS wrapper

Vault parses the `seal "awskms"` stanza, merges the supported AWS KMS environment variables, and passes the resulting map to the AWS KMS wrapper with environment lookup disabled inside the wrapper. Consequently, the source of truth for wrapper-specific stanza keys is the wrapper's `getOpts` implementation.

For this experiment the relevant accepted keys are `kms_key_id`, `region`, `shared_creds_filename`, `shared_creds_profile`, `role_arn`, and `role_session_name`. However, Vault 2.1.0+ent embeds `awskms/v2 v2.0.11` with `awsutil v0.3.0`. That credential chain appends the shared-credentials provider before the AssumeRole provider. Since AWS chains select the first valid provider, KMS receives the bootstrap user credentials and never reaches the assumed-role credentials. Therefore this exact combination accepts `role_arn` syntactically but does **not** provide the intended AssumeRole behavior when the source is a shared credentials profile.

> **Version-specific limitation:** the notebook keeps this configuration as a reproducible compatibility test. Do not grant KMS access directly to the bootstrap user to hide the result. For renewable production credentials, use the Web Identity/OIDC notebook. A pre-generated STS session can demonstrate temporary credentials, but Vault cannot refresh that externally generated session automatically.

`role_external_id` is deliberately **not** used here. Although the lower-level credential-chain helper can represent an external ID, the AWS KMS wrapper option parser does not expose a `role_external_id` field/key. An arbitrary key may survive the generic Vault parser, but that does not make it effective: the wrapper ignores what `getOpts` does not consume. Therefore this notebook's trust policy identifies the bootstrap user by ARN and does not require an ExternalId.

Environment merging supports `AWSKMS_WRAPPER_KEY_ID`, `AWS_ACCESS_KEY_ID`, `AWS_DEFAULT_REGION`, `AWS_KMS_ENDPOINT`, `AWS_REGION`, `AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`, and `VAULT_AWSKMS_SEAL_KEY_ID`. This notebook intentionally avoids credential environment variables inside the Vault Pods and mounts a shared credentials file instead.

Source references: [Vault Enterprise environment merge](https://github.com/hashicorp/vault-enterprise/blob/03515e06228c7eb8877e3043e87cd2ccf58d963a/internalshared/configutil/env_var_util.go#L25), [Enterprise AWS KMS wrapper construction](https://github.com/hashicorp/vault-enterprise/blob/03515e06228c7eb8877e3043e87cd2ccf58d963a/internalshared/configutil/kms_awskms_ent.go#L19), and [AWS wrapper option parsing](https://github.com/hashicorp/go-kms-wrapping/blob/d4ca45ec7310b5efea9a72993046cf896ff69550/wrappers/awskms/options.go#L50).

## 1. Load base AWS credentials

In [1]:
# Read aws credentials from csv file and set as environment variables
import csv, os
with open('vault_test_accessKeys.csv', 'r', encoding='utf-8-sig') as csvfile:
    reader = csv.DictReader(csvfile)
    creds = next(reader)
    access_key = creds['Access key ID'].strip()
    secret_key = creds['Secret access key'].strip()
    os.environ['AWS_ACCESS_KEY_ID'] = access_key
    os.environ['AWS_SECRET_ACCESS_KEY'] = secret_key
    # The CSV contains a long-lived IAM key; never combine it with a stale STS token.
    os.environ.pop('AWS_SESSION_TOKEN', None)
    os.environ.pop('AWS_SECURITY_TOKEN', None)
    print(f"Access Key: {access_key[:8]}...")
    print(f"Credentials loaded.")

Access Key: AKIAWZO6...
Credentials loaded.


## 2. Create KMS Key, bootstrap IAM user, and AssumeRole role

Vault runs **on-prem** but needs access to an AWS KMS key for auto-unseal.  
We create:
- A bootstrap **IAM user** with programmatic access and only `sts:AssumeRole` permission
- An **IAM role** trusted by that user and authorized to use the KMS key
- A **KMS key policy** that grants cryptographic operations to the role

Vault reads the user's key from a shared credentials profile, assumes the role through STS, and uses the temporary role credentials for auto-unseal.

In [2]:
import boto3
import json
import os
import time
from botocore.exceptions import ClientError

REGION = 'eu-west-3'
IAM_USER_NAME = 'vault-autounseal'
KMS_ALIAS = 'alias/vault-auto-unseal'
POLICY_NAME = 'vault-autounseal-kms-policy'
ROLE_NAME = 'vault-kms-unseal'
USER_ASSUME_POLICY_NAME = 'vault-assume-kms-role'

try:
    kms_client = boto3.client('kms', region_name=REGION)
    iam_client = boto3.client('iam')
    sts_client = boto3.client('sts')
    caller_identity = sts_client.get_caller_identity()
    account_id = caller_identity['Account']
    caller_arn = caller_identity['Arn']
    target_user_arn = f"arn:aws:iam::{account_id}:user/{IAM_USER_NAME}"
    if caller_arn == target_user_arn:
        raise RuntimeError(
            f'The provisioning credentials belong to {target_user_arn}. '
            'Use a separate administrator/provisioner identity; rotating the '
            'bootstrap user while authenticating as that user invalidates the run.'
        )

    # ─── 1. Create or reuse IAM User ───────────────────────────────────────
    try:
        iam_client.create_user(
            UserName=IAM_USER_NAME,
            Tags=[{'Key': 'Purpose', 'Value': 'vault-auto-unseal'}]
        )
        print(f"✓ IAM user '{IAM_USER_NAME}' created")
        time.sleep(30)
    except ClientError as e:
        if e.response['Error']['Code'] == 'EntityAlreadyExists':
            print(f"✓ IAM user '{IAM_USER_NAME}' already exists — reusing")
            # Delete old bootstrap keys. The guard above guarantees none is
            # the credential currently provisioning this environment.
            for key in iam_client.list_access_keys(UserName=IAM_USER_NAME)['AccessKeyMetadata']:
                iam_client.delete_access_key(UserName=IAM_USER_NAME, AccessKeyId=key['AccessKeyId'])
            print("  Old access keys deleted")
        else:
            raise

    ak_response = iam_client.create_access_key(UserName=IAM_USER_NAME)
    vault_access_key_id     = ak_response['AccessKey']['AccessKeyId']
    vault_secret_access_key = ak_response['AccessKey']['SecretAccessKey']
    print(f"✓ Access Key ID: {vault_access_key_id[:8]}...")

    # ─── 2. Create the role that Vault will assume through STS ──────────────
    user_arn = target_user_arn
    role_arn = f"arn:aws:iam::{account_id}:role/{ROLE_NAME}"
    # Using the account principal plus an exact ARN condition avoids failures
    # while a newly-created IAM user is still propagating.
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"AWS": f"arn:aws:iam::{account_id}:root"},
            "Action": "sts:AssumeRole",
            "Condition": {"ArnEquals": {"aws:PrincipalArn": user_arn}}
        }]
    }
    try:
        iam_client.create_role(
            RoleName=ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description='Role assumed by Vault for AWS KMS auto-unseal',
            MaxSessionDuration=3600,
            Tags=[{'Key': 'Purpose', 'Value': 'vault-auto-unseal'}]
        )
        print(f"✓ IAM role '{ROLE_NAME}' created")
        print("  Waiting for IAM role propagation...")
        time.sleep(30)
    except ClientError as e:
        if e.response['Error']['Code'] == 'EntityAlreadyExists':
            iam_client.update_assume_role_policy(
                RoleName=ROLE_NAME,
                PolicyDocument=json.dumps(trust_policy)
            )
            print(f"✓ IAM role '{ROLE_NAME}' already exists — trust updated")
        else:
            raise

    # The bootstrap user can only obtain temporary credentials for this role.
    iam_client.put_user_policy(
        UserName=IAM_USER_NAME,
        PolicyName=USER_ASSUME_POLICY_NAME,
        PolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Action": "sts:AssumeRole",
                "Resource": role_arn
            }]
        })
    )
    print("✓ Bootstrap user can assume the KMS role")

    # Remove direct KMS access left by an earlier notebook run.
    legacy_policy_arn = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"
    try:
        iam_client.detach_user_policy(UserName=IAM_USER_NAME, PolicyArn=legacy_policy_arn)
    except ClientError as e:
        if e.response['Error']['Code'] != 'NoSuchEntity':
            raise

    # ─── 3. Create KMS Key ─────────────────────────────────────────────────
    # The key policy grants access to the assumed role, never to the user.
    kms_key_id = None
    try:
        existing = kms_client.describe_key(KeyId=KMS_ALIAS)
        metadata = existing['KeyMetadata']
        kms_key_id  = metadata['KeyId']
        kms_key_arn = metadata['Arn']
        if metadata['KeyState'] == 'PendingDeletion':
            kms_client.cancel_key_deletion(KeyId=kms_key_id)
            kms_client.enable_key(KeyId=kms_key_id)
            print('  Pending KMS deletion cancelled and key enabled')
        elif metadata['KeyState'] == 'Disabled':
            kms_client.enable_key(KeyId=kms_key_id)
            print('  KMS key enabled')
        print(f"✓ KMS key already exists — reusing: {kms_key_id}")
    except ClientError as e:
        if e.response['Error']['Code'] == 'NotFoundException':
            key_response = kms_client.create_key(
                Description='Vault Auto-Unseal Key (on-prem Vault)',
                KeyUsage='ENCRYPT_DECRYPT',
                Origin='AWS_KMS',
                Policy=json.dumps({
                    "Version": "2012-10-17",
                    "Id": "vault-auto-unseal-key-policy",
                    # Create with a stable principal, then add the role below
                    # with retry logic for IAM propagation.
                    "Statement": [{
                        "Sid": "EnableRootAccountFullAccess",
                        "Effect": "Allow",
                        "Principal": {"AWS": f"arn:aws:iam::{account_id}:root"},
                        "Action": "kms:*",
                        "Resource": "*"
                    }]
                }),
                Tags=[{'TagKey': 'Purpose', 'TagValue': 'vault-auto-unseal'}]
            )
            kms_key_id  = key_response['KeyMetadata']['KeyId']
            kms_key_arn = key_response['KeyMetadata']['Arn']
            print(f"✓ KMS Key ID : {kms_key_id}")
            print(f"  KMS Key ARN: {kms_key_arn}")

            kms_client.create_alias(AliasName=KMS_ALIAS, TargetKeyId=kms_key_id)
            print(f"✓ KMS alias '{KMS_ALIAS}' created")
        else:
            raise

    # Replace any legacy user grant with an explicit grant to the role.
    key_policy = {
        "Version": "2012-10-17",
        "Id": "vault-auto-unseal-key-policy",
        "Statement": [
            {
                "Sid": "EnableRootAccountFullAccess",
                "Effect": "Allow",
                "Principal": {"AWS": f"arn:aws:iam::{account_id}:root"},
                "Action": "kms:*",
                "Resource": "*"
            },
            {
                "Sid": "AllowVaultAutoUnsealRole",
                "Effect": "Allow",
                "Principal": {"AWS": role_arn},
                "Action": ["kms:Encrypt", "kms:Decrypt", "kms:DescribeKey"],
                "Resource": "*"
            }
        ]
    }
    for attempt in range(1, 7):
        try:
            kms_client.put_key_policy(
                KeyId=kms_key_id, PolicyName='default',
                Policy=json.dumps(key_policy)
            )
            break
        except ClientError as e:
            if e.response['Error']['Code'] != 'MalformedPolicyDocumentException' or attempt == 6:
                raise
            print(f"  Waiting for KMS to recognize the role (attempt {attempt}/6)...")
            time.sleep(10)

    # ─── 4. Attach the KMS policy to the assumed role ───────────────────────
    kms_policy_document = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": ["kms:Encrypt", "kms:Decrypt", "kms:DescribeKey"],
            "Resource": kms_key_arn
        }]
    }

    kms_policy_arn = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"
    try:
        kms_policy_response = iam_client.create_policy(
            PolicyName=POLICY_NAME,
            Description='Allows KMS operations for Vault auto-unseal (on-prem Vault)',
            PolicyDocument=json.dumps(kms_policy_document)
        )
        kms_policy_arn = kms_policy_response['Policy']['Arn']
        print(f"✓ KMS policy created: {kms_policy_arn}")
    except ClientError as e:
        if e.response['Error']['Code'] == 'EntityAlreadyExists':
            # Reusing an ARN is not enough: update the document so it
            # always points to the current KMS key.
            versions = iam_client.list_policy_versions(PolicyArn=kms_policy_arn)['Versions']
            for version in versions:
                if not version['IsDefaultVersion']:
                    iam_client.delete_policy_version(
                        PolicyArn=kms_policy_arn, VersionId=version['VersionId']
                    )
            iam_client.create_policy_version(
                PolicyArn=kms_policy_arn,
                PolicyDocument=json.dumps(kms_policy_document),
                SetAsDefault=True
            )
            print(f"✓ KMS policy '{POLICY_NAME}' updated and reused")
        else:
            raise

    try:
        iam_client.attach_role_policy(RoleName=ROLE_NAME, PolicyArn=kms_policy_arn)
        print(f"✓ KMS policy attached to role")
    except ClientError:
        pass  # already attached

    # ─── 5. Validate AssumeRole and persist values ──────────────────────────
    bootstrap_sts = boto3.client(
        'sts',
        aws_access_key_id=vault_access_key_id,
        aws_secret_access_key=vault_secret_access_key
    )
    retryable_sts_errors = {'AccessDenied', 'InvalidClientTokenId'}
    for attempt in range(1, 13):
        try:
            assumed = bootstrap_sts.assume_role(
                RoleArn=role_arn,
                RoleSessionName='vault-auto-unseal-validation'
            )['Credentials']
            break
        except ClientError as e:
            error_code = e.response['Error']['Code']
            if error_code not in retryable_sts_errors or attempt == 12:
                raise
            print(
                f"  STS returned {error_code}; waiting for access-key/policy "
                f"propagation (attempt {attempt}/12)..."
            )
            time.sleep(10)
    boto3.client(
        'kms', region_name=REGION,
        aws_access_key_id=assumed['AccessKeyId'],
        aws_secret_access_key=assumed['SecretAccessKey'],
        aws_session_token=assumed['SessionToken']
    ).describe_key(KeyId=kms_key_id)
    os.environ['KMS_KEY_ID']                  = kms_key_id
    os.environ['REGION']                      = REGION
    os.environ['VAULT_AWS_ACCESS_KEY_ID']     = vault_access_key_id
    os.environ['VAULT_AWS_SECRET_ACCESS_KEY'] = vault_secret_access_key
    os.environ['VAULT_AWS_ROLE_ARN']           = role_arn

    print(f"\n✓ All values stored in environment")
    print(f"  KMS Key ARN: {kms_key_arn}")
    print(f"  Role ARN: {role_arn}")
    print("  ✓ AssumeRole + KMS access validated")
    print("  Bootstrap credentials will be mounted as a shared credentials file")

except ClientError as e:
    error_code = e.response['Error']['Code']
    error_msg  = e.response['Error']['Message']
    print(f"\n✗ AWS API error [{error_code}]: {error_msg}")
    print("  Resources are idempotently reused; cleanup is not required before retrying.")
    raise
except Exception as e:
    print(f"\n✗ Unexpected error: {e}")
    raise

✓ IAM user 'vault-autounseal' already exists — reusing
  Old access keys deleted
✓ Access Key ID: AKIAWZO6...
✓ IAM role 'vault-kms-unseal' already exists — trust updated
✓ Bootstrap user can assume the KMS role
✓ KMS key already exists — reusing: c2a15ec0-5f78-4651-86cd-6a23e55a6af2
✓ KMS policy 'vault-autounseal-kms-policy' already exists — reusing
✓ KMS policy attached to role

✗ AWS API error [InvalidClientTokenId]: The security token included in the request is invalid.
  You may need to clean up partially created resources before retrying.


ClientError: An error occurred (InvalidClientTokenId) when calling the AssumeRole operation: The security token included in the request is invalid.

## 3. Create K8S cluster and preload Vault Enterprise

The Vault Enterprise image is pulled by Podman on macOS, exported as a Docker archive, and loaded into the Minikube node. This avoids relying on Docker Hub DNS/connectivity from inside the Minikube VM. Helm uses `IfNotPresent`, so the preloaded image is selected without another registry pull.

In [4]:
! minikube delete -p workshop2

🔥  Deleting "PR" in docker ...
🔥  Removing /Users/jose/.minikube/machines/PR ...
💀  Removed all traces of the "PR" cluster.
🔥  Deleting "workshop" in docker ...
🔥  Removing /Users/jose/.minikube/machines/workshop ...
💀  Removed all traces of the "workshop" cluster.
🔥  Successfully deleted all profiles


In [5]:
! open -a Podman\ Desktop

In [6]:
import os
import pathlib
import re
import shutil
import subprocess

MINIKUBE_PROFILE = 'workshop2'
VAULT_IMAGE = 'docker.io/hashicorp/vault-enterprise:2.1.0-ent'
VAULT_IMAGE_ARCHIVE = pathlib.Path('/tmp/vault-enterprise-2.1.0-ent-assumerole.tar')

for binary in ('minikube', 'podman'):
    if not shutil.which(binary):
        raise RuntimeError(f'Required executable not found: {binary}')

subprocess.run(['minikube', 'start', '-p', MINIKUBE_PROFILE, '--force'], check=True)
subprocess.run(['kubectl', 'config', 'use-context', MINIKUBE_PROFILE], check=True)

# vfkit can advertise a gateway DNS server that is unreachable from the
# Minikube VM. Configure the VM with the active macOS resolvers, then
# restart CoreDNS so Vault Pods can resolve the regional STS/KMS endpoints.
macos_dns = subprocess.check_output(['scutil', '--dns'], text=True)
dns_servers = []
for address in re.findall(r'nameserver\[\d+\] : ([0-9.]+)', macos_dns):
    if address not in dns_servers:
        dns_servers.append(address)
if not dns_servers:
    raise RuntimeError('No active IPv4 DNS resolvers found in macOS')
dns_command = (
    'sudo resolvectl dns eth0 ' + ' '.join(dns_servers[:3]) +
    '; sudo resolvectl domain eth0 ~.'
)
subprocess.run([
    'minikube', 'ssh', '-p', MINIKUBE_PROFILE, '--', dns_command
], check=True)
subprocess.run([
    'kubectl', 'rollout', 'restart', 'deployment/coredns', '-n', 'kube-system'
], check=True)
subprocess.run([
    'kubectl', 'rollout', 'status', 'deployment/coredns', '-n', 'kube-system',
    '--timeout=120s'
], check=True)
aws_region = os.environ.get('REGION', 'eu-west-3')
kms_endpoint = f'kms.{aws_region}.amazonaws.com'
subprocess.run([
    'minikube', 'ssh', '-p', MINIKUBE_PROFILE, '--',
    f'resolvectl query {kms_endpoint}'
], check=True)
print(f'✓ Minikube DNS resolves {kms_endpoint}')

# Download on the host and preload the exact image used by Helm.
VAULT_IMAGE_ARCHIVE.unlink(missing_ok=True)
subprocess.run(['podman', 'pull', VAULT_IMAGE], check=True)
subprocess.run([
    'podman', 'save', '--format', 'docker-archive',
    '-o', str(VAULT_IMAGE_ARCHIVE), VAULT_IMAGE
], check=True)
subprocess.run([
    'minikube', 'image', 'load', '-p', MINIKUBE_PROFILE, str(VAULT_IMAGE_ARCHIVE)
], check=True)

loaded_images = subprocess.check_output(
    ['minikube', 'image', 'ls', '-p', MINIKUBE_PROFILE], text=True
)
if 'hashicorp/vault-enterprise:2.1.0-ent' not in loaded_images:
    raise RuntimeError('Vault Enterprise image was not loaded into Minikube')
print(f'✓ Preloaded {VAULT_IMAGE} into Minikube profile {MINIKUBE_PROFILE}')

😄  [workshop2] minikube v1.38.1 on Darwin 26.5.2 (arm64)
❗  minikube skips various validations when --force is supplied; this may lead to unexpected behavior
✨  Automatically selected the vfkit driver. Other choices: ssh, podman (experimental)
❗  Starting v1.39.0, minikube will default to "containerd" container runtime. See #21973 for more info.
👍  Starting "workshop2" primary control-plane node in "workshop2" cluster
🔥  Creating vfkit VM (CPUs=2, Memory=6144MB, Disk=20000MB) ...7m\
❗  Failing to connect to https://registry.k8s.io/ from inside the minikube VM
💡  To pull new external images, you may need to configure a proxy: https://minikube.sigs.k8s.io/docs/reference/networking/proxy/
🐳  Preparing Kubernetes v1.35.1 on Docker 28.5.2 ...7m\
🔗  Configuring bridge CNI (Container Networking Interface) ...?25h
🔎  Verifying Kubernetes components...
    ▪ Using image gcr.io/k8s-minikube/storage-provisioner:v5
🌟  Enabled addons: default-storageclass, storage-provisioner
🏄  Done! kubectl is no

In [7]:
%%bash
helm repo add hashicorp https://helm.releases.hashicorp.com
helm repo update

"hashicorp" already exists with the same configuration, skipping
Hang tight while we grab the latest from your chart repositories...
...Successfully got an update from the "secrets-store-csi-driver" chart repository
...Successfully got an update from the "kspm-helm-charts" chart repository
...Successfully got an update from the "hashicorp" chart repository
...Successfully got an update from the "prometheus-community" chart repository
...Successfully got an update from the "bitnami" chart repository
Update Complete. ⎈Happy Helming!⎈


In [8]:
%env WORKDIR=/tmp/vault
%env VAULT_K8S_NAMESPACE=vault
%env VAULT_HELM_RELEASE_NAME=vault
%env VAULT_SERVICE_NAME=vault-internal 
%env K8S_CLUSTER_NAME=cluster.local 

env: WORKDIR=/tmp/vault
env: VAULT_K8S_NAMESPACE=vault
env: VAULT_HELM_RELEASE_NAME=vault
env: VAULT_SERVICE_NAME=vault-internal
env: K8S_CLUSTER_NAME=cluster.local


In [9]:
%%bash
rm -rf /tmp/vault
mkdir /tmp/vault

In [10]:
! kubectl create namespace $VAULT_K8S_NAMESPACE

namespace/vault created


## 4. Create the AWS shared credentials Secret

Store the bootstrap user's access key in a Kubernetes Secret formatted as an AWS shared credentials file. Vault reads profile `vault-bootstrap` from the mounted file and uses it only to call `sts:AssumeRole`; the temporary role credentials are used for KMS.

In [ ]:
%%bash
# Create an AWS shared credentials file without exposing credentials in args.
mkdir -p "${WORKDIR}/aws"
umask 077
cat > "${WORKDIR}/aws/credentials" <<EOF
[vault-bootstrap]
aws_access_key_id = ${VAULT_AWS_ACCESS_KEY_ID}
aws_secret_access_key = ${VAULT_AWS_SECRET_ACCESS_KEY}
EOF

kubectl delete secret vault-aws-creds --namespace "${VAULT_K8S_NAMESPACE}" --ignore-not-found
kubectl create secret generic vault-aws-creds \
  --namespace "${VAULT_K8S_NAMESPACE}" \
  --from-file=credentials="${WORKDIR}/aws/credentials"

echo "✓ Secret 'vault-aws-creds' created in namespace '${VAULT_K8S_NAMESPACE}'"

## 5. Generate TLS certificates

In [12]:
%%bash

openssl genrsa -out ${WORKDIR}/vault.key 2048
cat > ${WORKDIR}/vault-csr.conf <<EOF
[req]
default_bits = 2048
prompt = no
encrypt_key = yes
default_md = sha256
distinguished_name = kubelet_serving
req_extensions = v3_req
[ kubelet_serving ]
O = system:nodes
CN = system:node:*.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
[ v3_req ]
basicConstraints = CA:FALSE
keyUsage = nonRepudiation, digitalSignature, keyEncipherment, dataEncipherment
extendedKeyUsage = serverAuth, clientAuth
subjectAltName = @alt_names
[alt_names]
DNS.1 = *.${VAULT_SERVICE_NAME}
DNS.2 = *.${VAULT_SERVICE_NAME}.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
DNS.3 = *.${VAULT_HELM_RELEASE_NAME}
DNS.4 = *.${VAULT_HELM_RELEASE_NAME}.svc.${K8S_CLUSTER_NAME}
IP.1 = 127.0.0.1
EOF

openssl req -new -key ${WORKDIR}/vault.key -out ${WORKDIR}/vault.csr -config ${WORKDIR}/vault-csr.conf


cat > ${WORKDIR}/csr.yaml <<EOF
apiVersion: certificates.k8s.io/v1
kind: CertificateSigningRequest
metadata:
   name: vault.svc
spec:
   signerName: kubernetes.io/kubelet-serving
   expirationSeconds: 8640000
   request: $(cat ${WORKDIR}/vault.csr|base64|tr -d '\n')
   usages:
   - digital signature
   - key encipherment
   - server auth
EOF

kubectl create -f ${WORKDIR}/csr.yaml
kubectl certificate approve vault.svc
kubectl get csr vault.svc
kubectl get csr vault.svc -o jsonpath='{.status.certificate}' | openssl base64 -d -A -out ${WORKDIR}/vault.crt
kubectl config view \
--raw \
--minify \
--flatten \
-o jsonpath='{.clusters[].cluster.certificate-authority-data}' \
| base64 -d > ${WORKDIR}/vault.ca



kubectl create secret generic vault-ha-tls \
   -n $VAULT_K8S_NAMESPACE \
   --from-file=vault.key=${WORKDIR}/vault.key \
   --from-file=vault.crt=${WORKDIR}/vault.crt \
   --from-file=vault.ca=${WORKDIR}/vault.ca

Generating RSA private key, 2048 bit long modulus
.....................................+++++
..........................+++++
e is 65537 (0x10001)


certificatesigningrequest.certificates.k8s.io/vault.svc created
certificatesigningrequest.certificates.k8s.io/vault.svc approved
NAME        AGE   SIGNERNAME                      REQUESTOR       REQUESTEDDURATION   CONDITION
vault.svc   0s    kubernetes.io/kubelet-serving   minikube-user   100d                Approved,Issued
secret/vault-ha-tls created


In [13]:
%%bash
secret=$(cat vault.hclic)
kubectl create secret generic vault-ent-license --from-literal="license=${secret}" -n $VAULT_K8S_NAMESPACE

secret/vault-ent-license created


## 6. Helm overrides — On-Prem Vault with AWS KMS auto-unseal

Key points:
- The Secret is mounted at `/vault/userconfig/aws/credentials`
- `shared_creds_profile` selects the `vault-bootstrap` profile
- `role_arn` makes Vault obtain and refresh temporary STS credentials
- The user can only assume the role; only the role can use KMS

In [ ]:
%%bash
cat > ${WORKDIR}/overrides.yaml <<EOF
global:
   enabled: true
   tlsDisable: false

csi:
   enabled: false

injector:
   enabled: false

logLevel: "trace"

server:
   image:
      repository: docker.io/hashicorp/vault-enterprise
      tag: 2.1.0-ent
      pullPolicy: IfNotPresent
   enterpriseLicense:
      secretName: vault-ent-license

   extraEnvironmentVars:
      VAULT_CACERT: /vault/userconfig/vault-ha-tls/vault.ca
      VAULT_TLSCERT: /vault/userconfig/vault-ha-tls/vault.crt
      VAULT_TLSKEY: /vault/userconfig/vault-ha-tls/vault.key

   volumes:
      - name: userconfig-vault-ha-tls
        secret:
         defaultMode: 420
         secretName: vault-ha-tls
      - name: aws-shared-credentials
        secret:
         defaultMode: 256
         secretName: vault-aws-creds

   volumeMounts:
      - mountPath: /vault/userconfig/vault-ha-tls
        name: userconfig-vault-ha-tls
        readOnly: true
      - mountPath: /vault/userconfig/aws
        name: aws-shared-credentials
        readOnly: true

   standalone:
      enabled: false
   affinity: ""
   ha:
      enabled: true
      replicas: 3
      raft:
         enabled: true
         setNodeId: true
         config: |
            ui = true
            listener "tcp" {
               tls_disable = 0
               address = "[::]:8200"
               cluster_address = "[::]:8201"
               tls_cert_file = "/vault/userconfig/vault-ha-tls/vault.crt"
               tls_key_file  = "/vault/userconfig/vault-ha-tls/vault.key"
               tls_client_ca_file = "/vault/userconfig/vault-ha-tls/vault.ca"
            }
            storage "raft" {
               path = "/vault/data"

               retry_join {
                  auto_join             = "provider=k8s namespace=vault label_selector=\"component=server,app.kubernetes.io/name=vault\""
                  auto_join_scheme      = "https"
                  leader_ca_cert_file   = "/vault/userconfig/vault-ha-tls/vault.ca"
                  leader_tls_servername = "vault-0.vault-internal"
               }

            }
            # The shared profile contains only bootstrap credentials. Vault's
            # AWS provider assumes the role and refreshes its STS credentials.
            seal "awskms" {
               region                = "${REGION}"
               kms_key_id            = "${KMS_KEY_ID}"
               shared_creds_filename = "/vault/userconfig/aws/credentials"
               shared_creds_profile  = "vault-bootstrap"
               role_arn              = "${VAULT_AWS_ROLE_ARN}"
               role_session_name     = "vault-auto-unseal"
            }
            telemetry {
               disable_hostname = true
               prometheus_retention_time = "12h"
            }
            disable_mlock = true
            service_registration "kubernetes" {}

   ui:
      enabled: true
      serviceType: "LoadBalancer"
      serviceNodePort: null
      externalPort: 8200

EOF

echo "✓ overrides.yaml written to ${WORKDIR}/overrides.yaml"
cat ${WORKDIR}/overrides.yaml

## 7. Deploy Vault with Helm

In [ ]:
%%bash
helm install ${VAULT_HELM_RELEASE_NAME} hashicorp/vault \
  --namespace ${VAULT_K8S_NAMESPACE} \
  --values ${WORKDIR}/overrides.yaml

echo ""
echo "✓ Helm release '${VAULT_HELM_RELEASE_NAME}' deployed"
echo "  Waiting for pods to be scheduled..."
kubectl get pods -n ${VAULT_K8S_NAMESPACE}

### Verify the shared credentials mount and STS/KMS initialization
Confirm the profile file is mounted without printing its contents, then inspect Vault startup logs for authentication or KMS errors.

In [ ]:
%%bash
# Wait for pod to be running first
kubectl wait pod vault-0 \
  --namespace ${VAULT_K8S_NAMESPACE} \
  --for=jsonpath='{.status.phase}'=Running \
  --timeout=120s

echo "=== AWS shared credentials mount ==="
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 -- \
  sh -c 'test -r /vault/userconfig/aws/credentials && echo "credentials profile mounted and readable"'

echo ""
echo "=== Vault logs (last 20 lines) ==="
kubectl logs vault-0 -n ${VAULT_K8S_NAMESPACE} --tail=40 | \
  grep -E -i 'seal|awskms|assume|sts|kms|error' || true

## 8. Initialize Vault

With AWS KMS auto-unseal, Vault only needs to be **initialized** once. The mounted bootstrap profile does not have KMS access: Vault uses it to call STS `AssumeRole`, and only the resulting temporary role credentials can use the KMS key. Recovery keys replace Shamir unseal keys for recovery operations.

In [ ]:
%%bash
# Give Vault a few seconds to finish starting its listener
sleep 5

echo "--- Initializing Vault (recovery-shares=1, recovery-threshold=1) ---"
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 -- \
  vault operator init \
  -recovery-shares=1 \
  -recovery-threshold=1 \
  -format=json > ${WORKDIR}/vault-init.json

echo ""
echo "✓ Init output saved securely to ${WORKDIR}/vault-init.json (not printed)"

In [ ]:
import json, os, re

try:
    with open('/tmp/vault/vault-init.json') as f:
        raw = f.read()

    if not raw.strip():
        raise ValueError("vault-init.json is empty – Vault init may have failed. Check the init cell output.")

    match = re.search(r'\{', raw)
    if match:
        raw = raw[match.start():]

    init_data = json.loads(raw)

    root_token   = init_data['root_token']
    recovery_key = init_data['recovery_keys_b64'][0]

    os.environ['VAULT_TOKEN'] = root_token
    print('✓ Root token loaded into VAULT_TOKEN without displaying it')
    print('✓ Recovery key present in the protected init file; store it securely')

except FileNotFoundError:
    print("✗ /tmp/vault/vault-init.json not found – run the Vault init cell first.")
except (json.JSONDecodeError, ValueError) as e:
    print(f"✗ Could not parse vault-init.json: {e}")
    print("  Re-run the Vault init cell and check its output for errors.")
except (KeyError, IndexError) as e:
    print(f"✗ Unexpected init JSON structure: {e}")
    raise

In [ ]:
%%bash
echo "=== Wait for the three-node Raft cluster ==="
kubectl wait pod -n ${VAULT_K8S_NAMESPACE} -l app.kubernetes.io/name=vault \
  --for=condition=Ready --timeout=240s
kubectl get pods -n ${VAULT_K8S_NAMESPACE} -l app.kubernetes.io/name=vault

echo "=== Restart vault-0 and prove credential reacquisition + auto-unseal ==="
kubectl delete pod -n ${VAULT_K8S_NAMESPACE} vault-0
kubectl wait pod/vault-0 -n ${VAULT_K8S_NAMESPACE} \
  --for=condition=Ready --timeout=240s
kubectl exec -n ${VAULT_K8S_NAMESPACE} vault-0 -- \
  vault status -tls-skip-verify
echo "✓ vault-0 restarted and became Ready without manual unseal"

# Clean up

In [ ]:
%%bash
# 1 – Complete local cleanup before touching AWS
helm uninstall ${VAULT_HELM_RELEASE_NAME} --namespace ${VAULT_K8S_NAMESPACE} || true
kubectl delete namespace ${VAULT_K8S_NAMESPACE} --ignore-not-found --wait=true --timeout=180s || true
minikube delete -p workshop2 || true
podman image rm --force docker.io/hashicorp/vault-enterprise:2.1.0-ent || true
rm -f /tmp/vault-enterprise-2.1.0-ent-assumerole.tar
rm -rf "${WORKDIR}"
echo "✓ Vault, namespace, Minikube profile, Podman image, and temporary secrets removed"

In [ ]:
# 2 – Remove all AWS resources created by this notebook
import boto3, os
from botocore.exceptions import ClientError

REGION        = os.environ.get('REGION', 'eu-west-3')
IAM_USER_NAME = 'vault-autounseal'
POLICY_NAME   = 'vault-autounseal-kms-policy'
KMS_ALIAS     = 'alias/vault-auto-unseal'

iam = boto3.client('iam')
kms = boto3.client('kms', region_name=REGION)
sts = boto3.client('sts')
account_id = sts.get_caller_identity()['Account']
kms_policy_arn = f"arn:aws:iam::{account_id}:policy/{POLICY_NAME}"

# Delete access keys for the user
try:
    for key in iam.list_access_keys(UserName=IAM_USER_NAME)['AccessKeyMetadata']:
        iam.delete_access_key(UserName=IAM_USER_NAME, AccessKeyId=key['AccessKeyId'])
    print("✓ Access keys deleted")
except ClientError as e:
    print(f"⚠ Access keys: {e.response['Error']['Message']}")

# Detach ALL attached policies from user (covers renamed/moved policies)
try:
    attached = iam.list_attached_user_policies(UserName=IAM_USER_NAME)['AttachedPolicies']
    for pol in attached:
        iam.detach_user_policy(UserName=IAM_USER_NAME, PolicyArn=pol['PolicyArn'])
        print(f"✓ Detached policy '{pol['PolicyName']}' from user")
    if not attached:
        print("  No attached policies found on user")
except ClientError as e:
    print(f"⚠ Detach policies: {e.response['Error']['Message']}")

# Delete user
try:
    for policy_name in iam.list_user_policies(UserName=IAM_USER_NAME)['PolicyNames']:
        iam.delete_user_policy(UserName=IAM_USER_NAME, PolicyName=policy_name)
        print(f"✓ Deleted inline user policy '{policy_name}'")
    iam.delete_user(UserName=IAM_USER_NAME)
    print(f"✓ IAM user '{IAM_USER_NAME}' deleted")
except ClientError as e:
    print(f"⚠ IAM user: {e.response['Error']['Message']}")

# Detach policy from ALL remaining entities, remove old versions, then delete
try:
    entities = iam.list_entities_for_policy(PolicyArn=kms_policy_arn)
    for u in entities.get('PolicyUsers', []):
        iam.detach_user_policy(UserName=u['UserName'], PolicyArn=kms_policy_arn)
        print(f"  Detached policy from user '{u['UserName']}'")
    for g in entities.get('PolicyGroups', []):
        iam.detach_group_policy(GroupName=g['GroupName'], PolicyArn=kms_policy_arn)
        print(f"  Detached policy from group '{g['GroupName']}'")
    for r in entities.get('PolicyRoles', []):
        iam.detach_role_policy(RoleName=r['RoleName'], PolicyArn=kms_policy_arn)
        print(f"  Detached policy from role '{r['RoleName']}'")
    for version in iam.list_policy_versions(PolicyArn=kms_policy_arn)['Versions']:
        if not version['IsDefaultVersion']:
            iam.delete_policy_version(
                PolicyArn=kms_policy_arn, VersionId=version['VersionId']
            )
    iam.delete_policy(PolicyArn=kms_policy_arn)
    print(f"✓ IAM policy '{POLICY_NAME}' deleted")
except ClientError as e:
    print(f"⚠ Policy: {e.response['Error']['Message']}")

# Delete the AssumeRole role after all managed and inline policies are removed
ROLE_NAME = 'vault-kms-unseal'
try:
    for policy_name in iam.list_role_policies(RoleName=ROLE_NAME)['PolicyNames']:
        iam.delete_role_policy(RoleName=ROLE_NAME, PolicyName=policy_name)
    for policy in iam.list_attached_role_policies(RoleName=ROLE_NAME)['AttachedPolicies']:
        iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn=policy['PolicyArn'])
    iam.delete_role(RoleName=ROLE_NAME)
    print(f"✓ IAM role '{ROLE_NAME}' deleted")
except ClientError as e:
    print(f"⚠ IAM role: {e.response['Error']['Message']}")

# Schedule KMS key deletion
try:
    alias_info = kms.describe_key(KeyId=KMS_ALIAS)
    kms_key_id = alias_info['KeyMetadata']['KeyId']
    kms.delete_alias(AliasName=KMS_ALIAS)
    kms.schedule_key_deletion(KeyId=kms_key_id, PendingWindowInDays=7)
    print(f"✓ KMS key '{kms_key_id}' scheduled for deletion in 7 days")
except ClientError as e:
    print(f"⚠ KMS key: {e.response['Error']['Message']}")

print("\n✓ AWS cleanup complete")

In [ ]:
%%bash
# Local cleanup intentionally runs before the AWS cleanup cell.
echo "✓ Cleanup order: local environment first, AWS resources second"